# HHP Aortic Diameter QC and Progression Analysis: Reproducibility Notebook

This notebook accompanies the supplemental methods for the HHP aortic diameter quality-control and progression analysis. It is designed as a **lightweight reproducibility notebook**: it does not re-run the full mixed-effects QC pipeline from raw clinical records. Instead, it uses the final de-identified analytic outputs generated by the publication pipeline to regenerate key plots, summary tables, and data checks.

The notebook supports review of four analytic domains:

1. long-format aortic measurements with QC flags;
2. patient-level OLS, mixed-effects, and best-model slope summaries;
3. progression categorization using prespecified slope thresholds;
4. high-risk participant ranking using aortic height index (AHI) and observed-minus-expected diameter.

All plots are regenerated from CSV files in the `data/` directory or, when run from the same folder as the uploaded files, from the current working directory.

## Supplementary Methods Narrative: Analytic Dataset and QC Overview

The analytic dataset was organized in long format, with each row corresponding to one aortic diameter measurement for a patient, an anatomic segment, and a study date. The required measurement-level variables were patient identifier, study date, height, source label, aortic segment, and diameter in centimeters. The pipeline derived follow-up time as years from each patient's baseline study, fit segment-specific mixed-effects models, calculated model residuals and interval growth rates, and flagged potentially implausible observations using both residual- and rate-based criteria.

For reproducibility, this notebook starts from the final analytic outputs after the QC pipeline has been executed. This approach avoids requiring reviewers to access protected raw clinical data while preserving the ability to regenerate the central analytic figures and verify the structure of the derived data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use default matplotlib style. Do not specify colors so figures remain portable.
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})

ROOT = Path(".")
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures"
TABLE_DIR = ROOT / "tables"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

def resolve_path(filename):
    """Find a required input file either in ./data or the current folder."""
    candidates = [DATA_DIR / filename, ROOT / filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Place it in ./data or next to this notebook.")

FILES = {
    "long": "aorta_data_long_with_outliers.csv",
    "slopes": "patient_slopes_ols_vs_mixed.csv",
    "progression": "patient_progression_categories.csv",
    "high_risk": "high_risk_combined_delta_AHI.csv",
    "ranked": "ranked_participants.csv",
    "qc_summary": "qc_summary_by_patient_segment.csv",
}

paths = {k: resolve_path(v) for k, v in FILES.items()}
paths

In [ ]:
# Load analytic outputs
long_df = pd.read_csv(paths["long"], parse_dates=["study_date"])
slopes_df = pd.read_csv(paths["slopes"])
progression_df = pd.read_csv(paths["progression"])
high_risk_df = pd.read_csv(paths["high_risk"])
ranked_df = pd.read_csv(paths["ranked"])
qc_summary_df = pd.read_csv(paths["qc_summary"])

# Basic normalization for common boolean columns that may be read as object
for df in [long_df, qc_summary_df]:
    for col in ["any_outlier_flag", "rate_review_flag", "outlier_rate_flag", "any_outlier"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1", "yes", "y"])

long_df["segment"] = long_df["segment"].astype(str).str.lower().str.strip()
slopes_df["segment"] = slopes_df["segment"].astype(str).str.lower().str.strip()
progression_df["segment"] = progression_df["segment"].astype(str).str.lower().str.strip()
high_risk_df["segment"] = high_risk_df["segment"].astype(str).str.lower().str.strip()
ranked_df["segment"] = ranked_df["segment"].astype(str).str.lower().str.strip()

{
    "measurement_rows": len(long_df),
    "unique_patients": long_df["patient_id"].nunique(),
    "segments": sorted(long_df["segment"].dropna().unique().tolist()),
    "slope_rows": len(slopes_df),
    "progression_rows": len(progression_df),
}

## Input Validation

The following checks confirm that all required columns are present. These are not statistical tests; they are reproducibility safeguards intended to fail early if the notebook is run against an incomplete export.

In [ ]:
required_columns = {
    "long_df": {
        "patient_id", "study_date", "height_cm", "Source", "segment", "diameter_cm",
        "time_years", "fitted_diameter_cm", "residual_cm", "resid_z",
        "rate_cm_per_year", "outlier_resid_flag", "outlier_rate_flag", "any_outlier_flag"
    },
    "slopes_df": {
        "patient_id", "segment", "n_echoes", "slope_cm_per_year",
        "ols_slope_cm_per_year", "best_model_slope_cm_per_year_pooled"
    },
    "progression_df": {
        "patient_id", "segment", "n_echoes", "progression_slope_cm_per_year",
        "progression_category", "high_confidence"
    },
    "high_risk_df": {
        "patient_id", "segment", "observed_cm", "expected_cm", "delta_cm",
        "height_cm", "AHI_cm_per_m", "combined_rank"
    },
    "ranked_df": {
        "patient_id", "segment", "AHI_cm_per_m", "delta_cm",
        "rank_size_then_growth"
    },
}

frames = {
    "long_df": long_df,
    "slopes_df": slopes_df,
    "progression_df": progression_df,
    "high_risk_df": high_risk_df,
    "ranked_df": ranked_df,
}

missing = {}
for name, cols in required_columns.items():
    missing[name] = sorted(cols - set(frames[name].columns))

pd.DataFrame({
    "dataset": list(missing.keys()),
    "missing_required_columns": [", ".join(v) if v else "None" for v in missing.values()]
})

## Cohort and QC Summary

Quality control was performed using two complementary strategies. First, mixed-effects residuals were standardized within aortic segment to identify measurements that were unusually discrepant from expected values after accounting for patient-level trajectories. Second, interval growth rates were calculated between consecutive measurements within each patient and segment; biologically implausible rapid growth or large negative changes were flagged for exclusion from downstream slope estimation.

The table below summarizes the measurement-level dataset and QC flags after execution of the full pipeline.

In [ ]:
summary_rows = []

for seg, g in long_df.groupby("segment"):
    summary_rows.append({
        "segment": seg,
        "n_measurements": len(g),
        "n_patients": g["patient_id"].nunique(),
        "n_flagged_any": int(g["any_outlier_flag"].sum()),
        "n_residual_suspect_or_outlier": int(g["outlier_resid_flag"].isin(["suspect", "outlier"]).sum()),
        "n_rate_outlier": int(g["outlier_rate_flag"].sum()),
        "median_measurements_per_patient": float(g.groupby("patient_id").size().median()),
    })

qc_overview = pd.DataFrame(summary_rows)
qc_overview.to_csv(TABLE_DIR / "qc_overview_by_segment.csv", index=False)
qc_overview

In [ ]:
# Figure 1: QC flag counts by segment
flag_counts = (
    long_df.assign(
        residual_flag=long_df["outlier_resid_flag"].isin(["suspect", "outlier"]),
        rate_flag=long_df["outlier_rate_flag"],
        any_flag=long_df["any_outlier_flag"],
    )
    .groupby("segment")[["residual_flag", "rate_flag", "any_flag"]]
    .sum()
)

ax = flag_counts.plot(kind="bar", figsize=(7, 4))
ax.set_title("QC flags by aortic segment")
ax.set_xlabel("Aortic segment")
ax.set_ylabel("Number of measurements")
ax.legend(["Residual-based flag", "Rate-based flag", "Any QC flag"], frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "qc_flags_by_segment.png")
plt.show()

## Representative Patient Trajectories

Patient-level trajectory plots were generated from the long-format measurement file. Each plot displays observed aortic diameters over time, the fitted mixed-effects trajectory, and QC-flagged measurements. Flagged measurements are shown but were excluded from downstream OLS and best-model slope estimation in the source pipeline. This convention preserves auditability while preventing flagged values from driving progression classification.

In [ ]:
def choose_representative_patients(progression_df, max_per_category=1):
    """Select representative patients across progression categories and segments."""
    reps = []
    priority = ["rapid", "mild", "stable", "uncertain"]
    for seg in sorted(progression_df["segment"].dropna().unique()):
        for cat in priority:
            sub = progression_df[
                (progression_df["segment"] == seg) &
                (progression_df["progression_category"] == cat)
            ].copy()
            if sub.empty:
                continue
            # Prefer high-confidence cases with more studies and larger absolute slope
            if "high_confidence" in sub.columns:
                sub["_hc"] = sub["high_confidence"].astype(str).str.lower().isin(["true", "1", "yes"])
            else:
                sub["_hc"] = False
            sub["_abs_slope"] = sub["progression_slope_cm_per_year"].abs()
            sub = sub.sort_values(["_hc", "n_echoes", "_abs_slope"], ascending=[False, False, False])
            reps.append((sub.iloc[0]["patient_id"], seg, cat))
            break
    return reps

representatives = choose_representative_patients(progression_df)
representatives

In [ ]:
def plot_patient_trajectory(patient_id, segment, save=True):
    sub = long_df[
        (long_df["patient_id"] == patient_id) &
        (long_df["segment"] == segment)
    ].copy().sort_values("time_years")

    if sub.empty:
        return None

    fig, ax = plt.subplots(figsize=(7, 4))

    usable = sub[~sub["any_outlier_flag"]]
    flagged = sub[sub["any_outlier_flag"]]

    # Observed points by source
    if "Source" in sub.columns:
        for src, g in usable.groupby("Source"):
            ax.scatter(g["time_years"], g["diameter_cm"], label=f"Observed: {src}", s=35)
    else:
        ax.scatter(usable["time_years"], usable["diameter_cm"], label="Observed", s=35)

    # Flagged points
    if not flagged.empty:
        ax.scatter(
            flagged["time_years"], flagged["diameter_cm"],
            marker="x", s=60, label="QC flagged"
        )

    # Mixed-effects fitted trajectory
    fit = sub.dropna(subset=["fitted_diameter_cm"])
    if len(fit) >= 2:
        ax.plot(fit["time_years"], fit["fitted_diameter_cm"], label="Mixed-effects fitted")

    # OLS line from usable values
    ols = usable.dropna(subset=["time_years", "diameter_cm"])
    if len(ols) >= 2 and ols["time_years"].nunique() >= 2:
        b, a = np.polyfit(ols["time_years"], ols["diameter_cm"], 1)
        grid = np.linspace(ols["time_years"].min(), ols["time_years"].max(), 100)
        ax.plot(grid, a + b * grid, linestyle="--", label="Per-patient OLS")

    ax.set_title(f"Representative trajectory: patient {patient_id}, {segment}")
    ax.set_xlabel("Years since baseline")
    ax.set_ylabel("Aortic diameter (cm)")
    ax.legend(frameon=False, loc="best")
    ax.grid(alpha=0.2)
    plt.tight_layout()

    if save:
        fig.savefig(FIG_DIR / f"trajectory_patient_{patient_id}_{segment}.png")
    return fig

for pid, seg, cat in representatives:
    print(f"Plotting patient {pid}, segment={seg}, category={cat}")
    plot_patient_trajectory(pid, seg)
    plt.show()

## Slope Estimation and Model Comparison

Patient-level slopes were summarized using several complementary estimates. Mixed-effects slopes describe patient-specific deviations from a segment-level trajectory, whereas per-patient OLS slopes provide a clinically interpretable growth rate based only on that patient's usable measurements. A best-model slope was also retained for sensitivity analysis, allowing conservative selection between linear and quadratic trajectories when the time series contained sufficient data.

For progression categorization, negative slopes were floored to zero for interpretability, and categories were assigned using prespecified thresholds: rapid progression for growth ≥0.30 cm/year, mild progression for 0.10 to <0.30 cm/year, stable for <0.10 cm/year, and uncertain when insufficient high-confidence data were available.

In [ ]:
# Figure 2: OLS vs mixed slope comparison
plot_df = slopes_df.dropna(subset=["ols_slope_cm_per_year", "slope_cm_per_year"]).copy()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for seg, g in plot_df.groupby("segment"):
    ax.scatter(g["ols_slope_cm_per_year"], g["slope_cm_per_year"], label=seg, alpha=0.75)

lims = [
    np.nanmin([plot_df["ols_slope_cm_per_year"].min(), plot_df["slope_cm_per_year"].min()]),
    np.nanmax([plot_df["ols_slope_cm_per_year"].max(), plot_df["slope_cm_per_year"].max()])
]
pad = (lims[1] - lims[0]) * 0.05 if lims[1] > lims[0] else 0.1
lims = [lims[0] - pad, lims[1] + pad]
ax.plot(lims, lims, linestyle="--", label="Identity line")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_title("Per-patient OLS versus mixed-effects slope")
ax.set_xlabel("OLS slope (cm/year)")
ax.set_ylabel("Mixed-effects slope (cm/year)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / "ols_vs_mixed_slope_scatter.png")
plt.show()

In [ ]:
# Figure 3: Distribution of progression slopes
dist_df = progression_df.dropna(subset=["progression_slope_cm_per_year"]).copy()

fig, ax = plt.subplots(figsize=(7, 4))
for seg, g in dist_df.groupby("segment"):
    ax.hist(g["progression_slope_cm_per_year"], bins=30, alpha=0.6, label=seg)

ax.axvline(0.10, linestyle="--", label="Mild threshold: 0.10 cm/year")
ax.axvline(0.30, linestyle=":", label="Rapid threshold: 0.30 cm/year")
ax.set_title("Distribution of progression slopes")
ax.set_xlabel("Progression slope (cm/year)")
ax.set_ylabel("Number of patient-segment observations")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "progression_slope_distribution.png")
plt.show()

In [ ]:
# Figure 4: Progression category counts
cat_order = ["stable", "mild", "rapid", "uncertain"]
cat_counts = (
    progression_df
    .assign(progression_category=pd.Categorical(progression_df["progression_category"], categories=cat_order, ordered=True))
    .groupby(["segment", "progression_category"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=cat_order)
)

ax = cat_counts.plot(kind="bar", figsize=(7, 4))
ax.set_title("Progression category by aortic segment")
ax.set_xlabel("Aortic segment")
ax.set_ylabel("Number of patient-segment observations")
ax.legend(title="Progression category", frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "progression_category_counts.png")
plt.show()

cat_counts.to_csv(TABLE_DIR / "progression_category_counts.csv")
cat_counts

## High-Risk Ranking: AHI and Observed-Minus-Expected Diameter

The high-risk ranking file identifies the maximum observed aortic diameter for each patient and segment, compares it with the mixed-effects expected diameter at the same timepoint, and calculates the aortic height index. Ranking prioritizes aortic size by AHI and then uses observed-minus-expected enlargement as a secondary criterion within near-tied AHI strata. This approach is intended for research prioritization and visualization, not for clinical decision-making.

In [ ]:
# Figure 5: Delta versus AHI scatter
hr = high_risk_df.dropna(subset=["delta_cm", "AHI_cm_per_m"]).copy()

fig, ax = plt.subplots(figsize=(7, 5))
for seg, g in hr.groupby("segment"):
    ax.scatter(g["delta_cm"], g["AHI_cm_per_m"], label=seg, alpha=0.75)

ax.axvline(0, linestyle="--", linewidth=1)
ax.set_title("Observed-minus-expected diameter versus AHI")
ax.set_xlabel("Observed – expected diameter (cm)")
ax.set_ylabel("Aortic height index (cm/m)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / "delta_vs_ahi_scatter.png")
plt.show()

In [ ]:
# Figure 6: Top-ranked participants by size-then-growth rank
top_n = 15
top_ranked = (
    ranked_df.sort_values(["segment", "rank_size_then_growth"])
    .groupby("segment")
    .head(top_n)
    .copy()
)

# Use a compact label to avoid exposing full identifiers in figure titles;
# this still uses the de-identified patient_id from the analytic dataset.
top_ranked["label"] = top_ranked["patient_id"].astype(str)

for seg, g in top_ranked.groupby("segment"):
    g = g.sort_values("rank_size_then_growth", ascending=True)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(g["label"], g["AHI_cm_per_m"])
    ax.invert_yaxis()
    ax.set_title(f"Top {top_n} ranked participants by AHI: {seg}")
    ax.set_xlabel("Aortic height index (cm/m)")
    ax.set_ylabel("De-identified patient ID")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"top_ranked_AHI_{seg}.png")
    plt.show()

## Source-Stratified Measurement Summary

Because measurements may originate from different sources, the QC summary retains source labels. The plot below summarizes the number of measurements by aortic segment and source. This figure is useful for documenting whether the analytic dataset is dominated by a single measurement source and for interpreting source-stratified sensitivity analyses.

In [ ]:
# Figure 7: Measurement source composition
if "Source" in long_df.columns:
    source_counts = (
        long_df.groupby(["segment", "Source"])
        .size()
        .unstack(fill_value=0)
    )
    ax = source_counts.plot(kind="bar", stacked=True, figsize=(7, 4))
    ax.set_title("Measurement source composition by segment")
    ax.set_xlabel("Aortic segment")
    ax.set_ylabel("Number of measurements")
    ax.legend(title="Source", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "source_composition_by_segment.png", bbox_inches="tight")
    plt.show()
    source_counts.to_csv(TABLE_DIR / "source_composition_by_segment.csv")
    display(source_counts)
else:
    print("Source column not available.")

## Exported Tables

The following reviewer-facing tables are exported to the `tables/` directory:

- `qc_overview_by_segment.csv`
- `progression_category_counts.csv`
- `source_composition_by_segment.csv` when source labels are available
- `top_ranked_participants_by_segment.csv`

These tables provide compact validation targets for the manuscript and supplement.

In [ ]:
top_ranked_table = (
    ranked_df.sort_values(["segment", "rank_size_then_growth"])
    .groupby("segment")
    .head(20)
    [["segment", "patient_id", "rank_size_then_growth", "AHI_cm_per_m", "delta_cm", "observed_cm", "expected_cm", "n_meas", "high_confidence"]]
)
top_ranked_table.to_csv(TABLE_DIR / "top_ranked_participants_by_segment.csv", index=False)
top_ranked_table.head(20)

## Reproducibility Notes and Limitations

This notebook regenerates key supplemental plots from de-identified derived analytic outputs. It is not intended to replace the primary Python pipeline used to create the analytic files. The primary pipeline should remain the authoritative source for mixed-effects model fitting, residual-based QC, rate-based outlier detection, best-model selection, and ranking. This separation allows the notebook to remain fast, transparent, and executable by reviewers without requiring protected raw clinical records.

All results generated here should be interpreted as research outputs. Automated QC flags, progression categories, and high-risk rankings are intended for cohort characterization, sensitivity analysis, and figure generation, not for clinical decision-making.

In [ ]:
print("Notebook completed successfully.")
print(f"Figures written to: {FIG_DIR.resolve()}")
print(f"Tables written to: {TABLE_DIR.resolve()}")